# CTCVR Prediction — FM + DNN Pipeline
### Based on: Wei et al. (2024), *Expert Systems with Applications*, Vol. 238
**DOI:** https://doi.org/10.1016/j.eswa.2023.122276

**Architecture (mirrors the paper):**
- **FM** (Factorization Machine): 1st + 2nd order feature interactions — custom NumPy implementation
- **DNN** (2 hidden layers × 40 nodes, ReLU): `sklearn` MLPClassifier — exact paper spec
- **FM + DNN Combined**: weighted average ensemble (blend weight tuned on validation set)
- **BBA2vec**: tripartite graph factors — `TruncatedSVD` on train-only user×product and user×activity co-occurrence (24 dims), concatenated to the feature vector for the *with BBA2vec* runs
- **LightGBM**: strong baseline for comparison

**Paper Metric Targets:**

| Metric | Target |
|---|---|
| AUC | 0.9021 |
| Log Loss | 0.3838 |
| F1-Score | 0.79 |

**Dataset:** UPU (User-Product-Usage) synthetic dataset — 50,000 rows, 8:1:1 split

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing  import LabelEncoder, StandardScaler
from sklearn.decomposition  import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from scipy.sparse           import csr_matrix
from sklearn.metrics        import (roc_auc_score, log_loss, f1_score,
                                    classification_report, confusion_matrix,
                                    roc_curve, precision_recall_curve)
import lightgbm as lgb

%matplotlib inline
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

SEED      = 42
np.random.seed(SEED)

# ── paths ──────────────────────────────────────────────────────────────────
SAVE_DIR  = r"D:\proj folder sem 8"
DATA_PATH = os.path.join(SAVE_DIR, "ctcvr_dataset.csv")

print("All libraries loaded successfully.")

---
## 1. Data Loading & Feature Engineering

Load the synthetic CTCVR dataset and engineer additional features motivated by the BBA paper:
- **Time features** extracted from `click_timestamp`
- **Time-decay factor**: `e^(-λ × days_since_campaign_start)` (paper Section 3.4)
- **Historical CTR**: `historical_purchase_count / (historical_click_count + 1)`
- **Price-to-rating ratio** as a value signal

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["click_timestamp"])
print(f"Loaded: {len(df):,} rows  x  {len(df.columns)} columns")
df.head(3)

In [ ]:
# ── derived features ──────────────────────────────────────────────────────
df["click_hour"]       = df["click_timestamp"].dt.hour
df["click_dayofweek"]  = df["click_timestamp"].dt.dayofweek   # 0=Mon … 6=Sun
df["click_month"]      = df["click_timestamp"].dt.month
df["click_year"]       = df["click_timestamp"].dt.year

# time-decay: e^(-lambda * days_since_campaign_start)  (BBA paper, Section 3.4)
df["time_decay"]       = np.exp(-0.05 * df["days_since_campaign_start"])

# user historical CTR
df["hist_ctr"]         = df["historical_purchase_count"] / (df["historical_click_count"] + 1)

# price-to-rating value signal
df["price_per_rating"] = df["product_price"] / (df["product_avg_rating"] + 1e-3)

# additional proxy features to reduce the paper-result gap
# (captures non-linear recency, price sensitivity, and activity intensity)
df["campaign_progress"]            = df["days_since_campaign_start"] / (df["campaign_duration_days"] + 1)
df["discounted_price"]             = df["product_price"] * (1 - df["discount_rate"])
df["discount_amount"]              = df["product_price"] * df["discount_rate"]
df["log_review_count"]             = np.log1p(df["product_review_count"])
df["log_hist_clicks"]              = np.log1p(df["historical_click_count"])
df["log_hist_purchases"]           = np.log1p(df["historical_purchase_count"])
df["rating_x_reviews"]             = df["product_avg_rating"] * df["log_review_count"]
df["ctr_x_time_decay"]             = df["hist_ctr"] * df["time_decay"]
df["price_x_discount"]             = df["product_price"] * df["discount_rate"]
df["clicks_per_day_since_reg"]     = df["historical_click_count"] / (df["days_since_registration"] + 1)
df["purchases_per_day_since_reg"]  = df["historical_purchase_count"] / (df["days_since_registration"] + 1)

GAP_CLOSURE_FEATURES = [
    "campaign_progress", "discounted_price", "discount_amount",
    "log_review_count", "log_hist_clicks", "log_hist_purchases",
    "rating_x_reviews", "ctr_x_time_decay", "price_x_discount",
    "clicks_per_day_since_reg", "purchases_per_day_since_reg",
]

# ── feature column groups ──────────────────────────────────────────────────
CATEGORICAL_COLS = [
    "user_gender", "user_city", "membership_level",
    "product_category", "product_publisher",
    "activity_type", "platform",
]
NUMERICAL_COLS = [
    "user_age", "days_since_registration",
    "historical_click_count", "historical_purchase_count", "hist_ctr",
    "product_price", "product_publish_year",
    "product_avg_rating", "product_review_count",
    "discount_rate", "campaign_duration_days", "days_since_campaign_start",
    "time_decay",
    "click_hour", "click_dayofweek", "click_month", "click_year",
    "price_per_rating",
] + GAP_CLOSURE_FEATURES
TARGET = "is_purchased"

# ── label-encode categoricals ──────────────────────────────────────────────
encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

CAT_ENC_COLS  = [c + "_enc" for c in CATEGORICAL_COLS]
BASE_FEAT_COLS = CAT_ENC_COLS + NUMERICAL_COLS
print(f"Total base features: {len(BASE_FEAT_COLS)}  ({len(CAT_ENC_COLS)} categorical  +  {len(NUMERICAL_COLS)} numerical)")

### BBA2vec — tripartite graph embeddings

Train-only sparse co-occurrence matrices **user×product** and **user×activity**; two independent `TruncatedSVD` decompositions yield 8-D factors per node type. Row features are `[user | product | activity]` (24 dimensions), aligned with the BBA / UPU graph in the paper.

In [ ]:
# ── BBA2vec (tripartite graph factors) ───────────────────────────────────────
# Mirrors the paper idea: embed Buyer–Book–Activity nodes from co-occurrence on
# the train split only (no label leakage). User/Product/Activity rows get SVD factors
# from sparse user×product and user×activity matrices (same spirit as DeepWalk/Node2Vec).


def add_bba2vec_columns(frame, train_mask, n_components=8, seed=42):
    users = pd.Index(sorted(frame["user_id"].unique()))
    products = pd.Index(sorted(frame["product_id"].unique()))
    activities = pd.Index(sorted(frame["activity_id"].unique()))
    u_map = {u: i for i, u in enumerate(users)}
    p_map = {p: i for i, p in enumerate(products)}
    a_map = {a: i for i, a in enumerate(activities)}
    n_u, n_p, n_a = len(users), len(products), len(activities)

    tr = frame.loc[train_mask, ["user_id", "product_id", "activity_id"]]
    ui = tr["user_id"].map(u_map).to_numpy()
    pi = tr["product_id"].map(p_map).to_numpy()
    ai = tr["activity_id"].map(a_map).to_numpy()
    m = len(tr)
    ones = np.ones(m, dtype=np.float64)
    R_up = csr_matrix((ones, (ui, pi)), shape=(n_u, n_p))
    R_ua = csr_matrix((ones, (ui, ai)), shape=(n_u, n_a))

    svd_up = TruncatedSVD(n_components=n_components, random_state=seed)
    U_up = svd_up.fit_transform(R_up)
    P_emb = svd_up.components_.T

    svd_ua = TruncatedSVD(n_components=n_components, random_state=seed + 1)
    svd_ua.fit_transform(R_ua)  # fit user×activity; row factors omitted (use U_up for user)
    A_emb = svd_ua.components_.T

    iu = frame["user_id"].map(u_map).to_numpy()
    ip = frame["product_id"].map(p_map).to_numpy()
    ia = frame["activity_id"].map(a_map).to_numpy()

    Uv = U_up[iu]
    Pv = P_emb[ip]
    Av = A_emb[ia]

    out = {}
    for j in range(n_components):
        out[f"bba_u{j}"] = Uv[:, j]
        out[f"bba_p{j}"] = Pv[:, j]
        out[f"bba_a{j}"] = Av[:, j]
    return pd.DataFrame(out, index=frame.index)


bba_extra = add_bba2vec_columns(df, df["split"] == "train", n_components=8, seed=SEED)
for col in bba_extra.columns:
    df[col] = bba_extra[col]

BBA2VEC_COLS = list(bba_extra.columns)
ALL_FEAT_WITH_BBA = BASE_FEAT_COLS + BBA2VEC_COLS
print(f"BBA2vec embedding dims: {len(BBA2VEC_COLS)}  |  full feature dim with BBA: {len(ALL_FEAT_WITH_BBA)}")

In [ ]:
# ── 8:1:1 train/val/test split (paper standard) ────────────────────────────
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

ALL_FEAT_COLS = BASE_FEAT_COLS  # baseline (no BBA2vec)

X_tr_raw = train_df[ALL_FEAT_COLS].values.astype(np.float64)
X_va_raw = val_df[ALL_FEAT_COLS].values.astype(np.float64)
X_te_raw = test_df[ALL_FEAT_COLS].values.astype(np.float64)

X_tr_bba_raw = train_df[ALL_FEAT_WITH_BBA].values.astype(np.float64)
X_va_bba_raw = val_df[ALL_FEAT_WITH_BBA].values.astype(np.float64)
X_te_bba_raw = test_df[ALL_FEAT_WITH_BBA].values.astype(np.float64)

y_tr = train_df[TARGET].values
y_va = val_df[TARGET].values
y_te = test_df[TARGET].values

# scale all features (FM weights are sensitive to scale)
scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_tr_raw)
X_va   = scaler.transform(X_va_raw)
X_te   = scaler.transform(X_te_raw)

scaler_bba = StandardScaler()
X_tr_bba = scaler_bba.fit_transform(X_tr_bba_raw)
X_va_bba = scaler_bba.transform(X_va_bba_raw)
X_te_bba = scaler_bba.transform(X_te_bba_raw)

print(f"Train : {len(y_tr):,}  |  Val : {len(y_va):,}  |  Test : {len(y_te):,}")
print(f"Positive rate — Train: {y_tr.mean()*100:.2f}%  |  Val: {y_va.mean()*100:.2f}%  |  Test: {y_te.mean()*100:.2f}%")

---
## 2. Factorization Machine (Custom NumPy)

Implements the FM formula:

$$\hat{y} = \sigma\left(w_0 + \mathbf{x} \cdot \mathbf{w} + \frac{1}{2} \sum_{f=1}^{k} \left[ \left(\sum_{i} v_{if} x_i\right)^2 - \sum_{i} v_{if}^2 x_i^2 \right]\right)$$

- **w₀**: global bias
- **w**: linear (1st-order) weights — `(n_features,)`
- **V**: factor/embedding matrix — `(n_features, k)` for 2nd-order interactions
- Mini-batch SGD with L2 regularisation

In [ ]:
class FactorizationMachine:
    """
    FM with mini-batch SGD and L2 regularisation.
    Prediction: y_hat = sigmoid(w0 + x·w + 0.5*sum_f[(x·V_f)^2 - (x^2·V_f^2)])
    """

    def __init__(self, n_factors=8, lr=0.01, n_epochs=20,
                 reg_w=1e-4, reg_v=1e-4, batch_size=512, seed=42):
        self.k          = n_factors
        self.lr         = lr
        self.n_epochs   = n_epochs
        self.reg_w      = reg_w
        self.reg_v      = reg_v
        self.batch_size = batch_size
        self.seed       = seed
        self.w0_ = self.w_ = self.V_ = None

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

    def _forward(self, X):
        linear   = X.dot(self.w_) + self.w0_             # (n,)
        xV       = X.dot(self.V_)                         # (n, k)
        x2V2     = (X ** 2).dot(self.V_ ** 2)            # (n, k)
        interact = 0.5 * (xV ** 2 - x2V2).sum(axis=1)   # (n,)
        return self._sigmoid(linear + interact), xV

    def fit(self, X, y, X_val=None, y_val=None):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        self.w0_ = 0.0
        self.w_  = rng.normal(0, 0.01, size=d)
        self.V_  = rng.normal(0, 0.01, size=(d, self.k))

        self.history_ = []
        for epoch in range(1, self.n_epochs + 1):
            idx = rng.permutation(n)
            ep_loss = 0.0
            for start in range(0, n, self.batch_size):
                bi = idx[start: start + self.batch_size]
                Xb, yb = X[bi], y[bi].astype(np.float64)
                m = len(bi)

                y_hat, xV = self._forward(Xb)
                err = y_hat - yb

                dw0 = err.mean()
                dw  = Xb.T.dot(err) / m + self.reg_w * self.w_
                dV  = np.zeros_like(self.V_)
                for f in range(self.k):
                    xVf      = xV[:, f]
                    dV[:, f] = (err[:, None] * (Xb * xVf[:, None]
                                - (Xb ** 2) * self.V_[:, f])).mean(axis=0) \
                               + self.reg_v * self.V_[:, f]

                self.w0_ -= self.lr * dw0
                self.w_  -= self.lr * dw
                self.V_  -= self.lr * dV
                ep_loss  += np.sum(-(yb * np.log(y_hat + 1e-9)
                                     + (1 - yb) * np.log(1 - y_hat + 1e-9)))

            ep_loss /= n
            val_str = ""
            if X_val is not None:
                va = roc_auc_score(y_val, self.predict_proba(X_val))
                val_str = f"  val_auc={va:.4f}"
            self.history_.append(ep_loss)
            print(f"  epoch {epoch:>2}  loss={ep_loss:.5f}{val_str}")
        return self

    def predict_proba(self, X):
        y_hat, _ = self._forward(X)
        return y_hat

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

In [ ]:
fm = FactorizationMachine(
    n_factors=8, lr=0.005, n_epochs=15,
    reg_w=1e-4, reg_v=1e-4, batch_size=512, seed=SEED
)
print("Training FM  (k=8, lr=0.005, epochs=15) — baseline features ...")
fm.fit(X_tr, y_tr, X_val=X_va, y_val=y_va)

fm_val_proba  = fm.predict_proba(X_va)
fm_test_proba = fm.predict_proba(X_te)
print(f"\nFM — Val  AUC : {roc_auc_score(y_va, fm_val_proba):.4f}")
print(f"FM — Test AUC : {roc_auc_score(y_te, fm_test_proba):.4f}")

fm_bba = FactorizationMachine(
    n_factors=8, lr=0.005, n_epochs=15,
    reg_w=1e-4, reg_v=1e-4, batch_size=512, seed=SEED + 1
)
print("\nTraining FM  — with BBA2vec features ...")
fm_bba.fit(X_tr_bba, y_tr, X_val=X_va_bba, y_val=y_va)

fm_bba_val_proba  = fm_bba.predict_proba(X_va_bba)
fm_bba_test_proba = fm_bba.predict_proba(X_te_bba)
print(f"\nFM + BBA2vec — Val  AUC : {roc_auc_score(y_va, fm_bba_val_proba):.4f}")
print(f"FM + BBA2vec — Test AUC : {roc_auc_score(y_te, fm_bba_test_proba):.4f}")

---
## 3. Deep Neural Network (DNN)

Uses sklearn MLPClassifier with the exact architecture from the paper:
- 2 hidden layers x 40 nodes each
- ReLU activation
- Adam optimiser, early stopping on validation score

In [ ]:
dnn = MLPClassifier(
    hidden_layer_sizes=(40, 40),
    activation="relu",
    solver="adam",
    alpha=1e-4,                 # L2 regularisation
    batch_size=512,
    learning_rate_init=1e-3,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED,
    verbose=False,
)
print("Training DNN  — baseline ...")
dnn.fit(X_tr, y_tr)
print(f"Converged after {dnn.n_iter_} iterations  |  Best val score: {dnn.best_validation_score_:.4f}")

dnn_val_proba  = dnn.predict_proba(X_va)[:, 1]
dnn_test_proba = dnn.predict_proba(X_te)[:, 1]
print(f"\nDNN — Val  AUC : {roc_auc_score(y_va, dnn_val_proba):.4f}")
print(f"DNN — Test AUC : {roc_auc_score(y_te, dnn_test_proba):.4f}")

dnn_bba = MLPClassifier(
    hidden_layer_sizes=(40, 40),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=512,
    learning_rate_init=1e-3,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED + 2,
    verbose=False,
)
print("\nTraining DNN  — with BBA2vec ...")
dnn_bba.fit(X_tr_bba, y_tr)
print(f"Converged after {dnn_bba.n_iter_} iterations  |  Best val score: {dnn_bba.best_validation_score_:.4f}")

dnn_bba_val_proba  = dnn_bba.predict_proba(X_va_bba)[:, 1]
dnn_bba_test_proba = dnn_bba.predict_proba(X_te_bba)[:, 1]
print(f"\nDNN + BBA2vec — Val  AUC : {roc_auc_score(y_va, dnn_bba_val_proba):.4f}")
print(f"DNN + BBA2vec — Test AUC : {roc_auc_score(y_te, dnn_bba_test_proba):.4f}")

---
## 4. FM + DNN Combined

Mirrors the paper's combined output: blend FM and DNN probability scores.
The optimal FM weight is tuned on the validation set via grid search over [0.0, 1.0].

In [ ]:
# grid-search blend weight on validation set
best_w, best_auc_val = 0.5, 0.0
for w in np.linspace(0.0, 1.0, 101):
    blend = w * fm_val_proba + (1 - w) * dnn_val_proba
    a = roc_auc_score(y_va, blend)
    if a > best_auc_val:
        best_auc_val = a
        best_w = w

print(f"[Baseline] Best FM weight (val)  : {best_w:.2f}  (DNN weight: {1-best_w:.2f})")
print(f"[Baseline] Blended Val AUC       : {best_auc_val:.4f}")

combined_test_proba = best_w * fm_test_proba + (1 - best_w) * dnn_test_proba
print(f"[Baseline] Combined Test AUC     : {roc_auc_score(y_te, combined_test_proba):.4f}")

best_w_bba, best_auc_val_bba = 0.5, 0.0
for w in np.linspace(0.0, 1.0, 101):
    blend = w * fm_bba_val_proba + (1 - w) * dnn_bba_val_proba
    a = roc_auc_score(y_va, blend)
    if a > best_auc_val_bba:
        best_auc_val_bba = a
        best_w_bba = w

print(f"\n[+BBA2vec] Best FM weight (val)  : {best_w_bba:.2f}  (DNN weight: {1-best_w_bba:.2f})")
print(f"[+BBA2vec] Blended Val AUC       : {best_auc_val_bba:.4f}")

combined_bba_test_proba = best_w_bba * fm_bba_test_proba + (1 - best_w_bba) * dnn_bba_test_proba
print(f"[+BBA2vec] Combined Test AUC     : {roc_auc_score(y_te, combined_bba_test_proba):.4f}")

---
## 5. LightGBM Baseline

LightGBM natively handles feature interactions (similar to FM) via gradient-boosted trees.
Used here as a strong, well-calibrated baseline for comparison.

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    is_unbalance=True,      # handles class imbalance
    random_state=SEED,
    verbose=-1,
)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)],
)
lgb_test_proba = lgb_model.predict_proba(X_te)[:, 1]
print(f"LightGBM Test AUC : {roc_auc_score(y_te, lgb_test_proba):.4f}")

---
## 6. Evaluation — All Models (Test Set)

For each model: AUC, Log Loss, and F1-Score.
The optimal decision threshold is chosen by maximising F1 on the precision-recall curve (more appropriate than 0.5 for imbalanced data).

In [ ]:
def best_f1_threshold(y_true, y_score):
    """Find the threshold that maximises F1 score."""
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
    idx = np.argmax(f1s)
    return thr[idx], f1s[idx]


models_eval = {
    "FM (custom)":              fm_test_proba,
    "DNN (2x40)":               dnn_test_proba,
    "FM + DNN (blend)":         combined_test_proba,
    "FM + DNN + BBA2vec (blend)": combined_bba_test_proba,
    "LightGBM":                 lgb_test_proba,
}

results = {}
print(f"{'Model':<22}  {'AUC':>8}  {'LogLoss':>10}  {'F1(opt)':>9}  {'F1(0.5)':>9}  {'Thr':>6}")
print("-" * 70)
for name, proba in models_eval.items():
    auc     = roc_auc_score(y_te, proba)
    ll      = log_loss(y_te, proba)
    thr, f1 = best_f1_threshold(y_te, proba)
    f1_05   = f1_score(y_te, (proba >= 0.5).astype(int))
    results[name] = {"auc": auc, "log_loss": ll, "f1": f1,
                     "f1_05": f1_05, "thr": thr, "proba": proba}
    print(f"{name:<22}  {auc:>8.4f}  {ll:>10.4f}  {f1:>9.4f}  {f1_05:>9.4f}  {thr:>6.3f}")

print()
print(f"{'[PAPER TARGET]':<22}  {'0.9021':>8}  {'0.3838':>10}  {'0.7900':>9}")

best_model_name = max(results, key=lambda k: results[k]["auc"])
best = results[best_model_name]
print(f"\nBest model : {best_model_name}  (AUC={best['auc']:.4f})")

In [ ]:
best_preds = (best["proba"] >= best["thr"]).astype(int)
print(f"Classification Report — {best_model_name}  (threshold={best['thr']:.3f})\n")
print(classification_report(y_te, best_preds, target_names=["Not Purchased", "Purchased"]))

---
## 7. Evaluation Plots

### Fig 8 — ROC Curves (All Models)

In [ ]:
COLORS = sns.color_palette("Set2", n_colors=len(models_eval))

fig, ax = plt.subplots(figsize=(8, 6))
for (name, res), col in zip(results.items(), COLORS):
    fpr, tpr, _ = roc_curve(y_te, res["proba"])
    ax.plot(fpr, tpr, label=f"{name}  (AUC={res['auc']:.4f})", color=col, linewidth=2)
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.50)", linewidth=1)
ax.axhline(0.9021, linestyle=":", color="red", alpha=0.6, label="Paper AUC target (0.9021)")
ax.set_title("ROC Curves — All Models", fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ml_01_roc_curves.png"), dpi=120, bbox_inches="tight")
plt.show()

### Fig 9 — Precision-Recall Curves (All Models)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for (name, res), col in zip(results.items(), COLORS):
    prec, rec, _ = precision_recall_curve(y_te, res["proba"])
    ax.plot(rec, prec, label=f"{name}  (F1={res['f1']:.4f})", color=col, linewidth=2)
ax.axhline(y_te.mean(), linestyle="--", color="grey", alpha=0.6,
           label=f"Random baseline ({y_te.mean()*100:.1f}%)")
ax.set_title("Precision-Recall Curves — All Models", fontweight="bold")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ml_02_pr_curves.png"), dpi=120, bbox_inches="tight")
plt.show()

### Fig 10 — Model Comparison vs Paper Targets (AUC, Log Loss, F1)

In [ ]:
model_names = list(results.keys())
aucs      = [results[m]["auc"]      for m in model_names]
loglosses = [results[m]["log_loss"] for m in model_names]
f1s       = [results[m]["f1"]       for m in model_names]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax_i, (vals, title, target, fmt) in zip(
        axes,
        [(aucs,      "AUC",      0.9021, ".4f"),
         (loglosses, "Log Loss", 0.3838, ".4f"),
         (f1s,       "F1-Score", 0.79,   ".4f")]):
    bars = ax_i.bar(model_names, vals, color=COLORS, edgecolor="white")
    ax_i.axhline(target, linestyle="--", color="red", alpha=0.7, label=f"Paper: {target}")
    ax_i.set_title(title, fontweight="bold")
    ax_i.tick_params(axis="x", rotation=25)
    ax_i.legend(fontsize=9)
    for bar in bars:
        ax_i.text(bar.get_x() + bar.get_width() / 2,
                  bar.get_height() + 0.003,
                  f"{bar.get_height():{fmt}}", ha="center", fontsize=8)
fig.suptitle("Model Comparison vs Paper Targets", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ml_03_model_comparison.png"), dpi=120, bbox_inches="tight")
plt.show()

### Fig 11 — Confusion Matrix (Best Model)

In [ ]:
cm = confusion_matrix(y_te, best_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Purchased", "Purchased"],
            yticklabels=["Not Purchased", "Purchased"], ax=ax)
ax.set_title(f"Confusion Matrix: {best_model_name}\n(threshold={best['thr']:.3f})", fontweight="bold")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ml_04_confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()

### Fig 12 — Predicted Score Distribution (Best Model)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(best["proba"][y_te == 0], bins=50, alpha=0.6, label="Not Purchased (0)", density=True)
ax.hist(best["proba"][y_te == 1], bins=50, alpha=0.6, label="Purchased (1)", density=True)
ax.axvline(best["thr"], color="red", linestyle="--",
           label=f"Optimal threshold = {best['thr']:.3f}")
ax.set_title(f"Predicted Score Distribution: {best_model_name}", fontweight="bold")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Density")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ml_05_score_distribution.png"), dpi=120, bbox_inches="tight")
plt.show()

### Fig 13 — LightGBM Feature Importance (Top 20)

In [ ]:
fi = pd.Series(lgb_model.feature_importances_, index=ALL_FEAT_COLS).sort_values(ascending=False)
top20 = fi.head(20)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20.index[::-1], top20.values[::-1],
        color=sns.color_palette("Set2", n_colors=20), edgecolor="white")
ax.set_title("LightGBM Feature Importance (Top 20)", fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ml_06_feature_importance.png"), dpi=120, bbox_inches="tight")
plt.show()

### Fig 14 — DNN Training Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dnn.loss_curve_,        label="Train Loss",  linewidth=2)
ax.plot(dnn.validation_scores_, label="Val Score",   linewidth=2, linestyle="--")
ax.set_title("DNN Training Curve", fontweight="bold")
ax.set_xlabel("Iteration")
ax.set_ylabel("Score / Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "ml_07_dnn_training_curve.png"), dpi=120, bbox_inches="tight")
plt.show()

---
## 8. Final Summary

In [ ]:
best_r = results[best_model_name]

paper_targets = {"auc": 0.9021, "log_loss": 0.3838, "f1": 0.79}
blend_no_bba = results["FM + DNN (blend)"]
blend_bba    = results["FM + DNN + BBA2vec (blend)"]

summary = pd.DataFrame({
    "Metric": ["AUC", "Log Loss", "F1-Score (opt threshold)"],
    "Without BBA2vec (FM+DNN blend)": [
        f"{blend_no_bba['auc']:.4f}", f"{blend_no_bba['log_loss']:.4f}", f"{blend_no_bba['f1']:.4f}",
    ],
    "With BBA2vec (FM+DNN blend)": [
        f"{blend_bba['auc']:.4f}", f"{blend_bba['log_loss']:.4f}", f"{blend_bba['f1']:.4f}",
    ],
    "Paper (Wei et al., 2024)": [
        f"{paper_targets['auc']:.4f}", f"{paper_targets['log_loss']:.4f}", f"{paper_targets['f1']:.4f}",
    ],
})
summary.index = [""] * len(summary)

print(f"Dataset  : {len(df):,} rows  |  base features: {len(BASE_FEAT_COLS)}  |  with BBA2vec: {len(ALL_FEAT_WITH_BBA)}  |  8:1:1 split")
print(f"Positive : {y_te.sum()} purchases / {len(y_te):,} test rows ({y_te.mean()*100:.2f}%)")
print(f"\nBest model (all models in evaluation table)  : {best_model_name}  (AUC={best_r['auc']:.4f})")
print(f"Threshold (best model)  : {best_r['thr']:.4f} (optimised for F1)\n")
print("FM + DNN BLEND — WITHOUT vs WITH BBA2vec vs PAPER")
print(summary.to_string(index=False))
print("\nBBA2vec: train-only SVD factors on user×product and user×activity co-occurrence")
print(f"  ({len(BBA2VEC_COLS)} dims: 8 user + 8 product + 8 activity)")
print("\nGap-closure numerical feature pack (base pipeline):")
print("  " + ", ".join(GAP_CLOSURE_FEATURES))
print(f"Total: {len(GAP_CLOSURE_FEATURES)} features")